##Install the packages

In [2]:
!uv add langchain==0.3.27 langchain-community==0.3.27 langchain-huggingface==0.3.1 langchain-core==0.3.74 langchain-chroma==0.2.5 pypdf -q

##Load Dependencies

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

## 1) Load the PDF  (one Document per page)




In [4]:
loader = PyPDFLoader("./data/langchain_demo.pdf")
docs = loader.load()
docs[0]

Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2026-02-02T21:45:08+00:00', 'source': './data/langchain_demo.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='LangChain Document Loaders - Demo Document\nUnderstanding LangChain Document Loaders\n1. Introduction\nLangChain provides powerful document loaders that allow you to ingest data from various sources into your\nLLM applications. Document loaders are essential for building RAG (Retrieval-Augmented Generation)\nsystems, chatbots, and knowledge bases.\nThis document serves as a demo file to test PDF loading capabilities in LangChain. When loaded, this\ncontent will be split into chunks and can be used for vector storage and retrieval.\n2. Types of Document Loaders\n  PyPDFLoader: Load PDF files page by page with metadata\n  TextLoader: Load plain text files (.txt)\n  CSVLoader: Load CSV files with row-based documents\n  JSONLoader: Load JSON files with jq-style extraction\n  Unstructure

## 2) Split into overlapping chunks

In [5]:
from h11._abnf import chunk_size
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap = 150)
chunks = splitter.split_documents(docs)
print(f"{len(chunks)} chuck created")


5 chuck created


##3) Embed & Store in Chroma

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# 3) Create the embedding function
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')


# 4) Embed all chunks and persist to Chroma
vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="handbook",
    persist_directory="./chroma_db")

print("Stored", vectordb._collection.count(),
      "vectors")


Stored 15 vectors


##4) Query the Vector Store

In [7]:
query = "What is the types of document loaders"
chunks_query = vectordb.similarity_search(query, k=3)
for i,d in enumerate(chunks_query) : 
    print(f"\n chuck {i} \n")
    print(d.metadata["page"],d.page_content)


 chuck 0 

0 LangChain Document Loaders - Demo Document
Understanding LangChain Document Loaders
1. Introduction
LangChain provides powerful document loaders that allow you to ingest data from various sources into your
LLM applications. Document loaders are essential for building RAG (Retrieval-Augmented Generation)
systems, chatbots, and knowledge bases.
This document serves as a demo file to test PDF loading capabilities in LangChain. When loaded, this
content will be split into chunks and can be used for vector storage and retrieval.
2. Types of Document Loaders
  PyPDFLoader: Load PDF files page by page with metadata
  TextLoader: Load plain text files (.txt)
  CSVLoader: Load CSV files with row-based documents
  JSONLoader: Load JSON files with jq-style extraction
  UnstructuredLoader: Handle various file formats automatically
  DirectoryLoader: Load all files from a directory
  WebBaseLoader: Scrape and load web pages
  YouTubeLoader: Extract transcripts from YouTube videos

 ch

In [8]:
# Plug into a RAG chain as a retriever
retriever = vectordb.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
    )

results = retriever.get_relevant_documents(query)
for i, r in enumerate(results):
    print(f"\n chunk : {i} \n")
    print(r.page_content, r.metadata)


 chunk : 0 

LangChain Document Loaders - Demo Document
Understanding LangChain Document Loaders
1. Introduction
LangChain provides powerful document loaders that allow you to ingest data from various sources into your
LLM applications. Document loaders are essential for building RAG (Retrieval-Augmented Generation)
systems, chatbots, and knowledge bases.
This document serves as a demo file to test PDF loading capabilities in LangChain. When loaded, this
content will be split into chunks and can be used for vector storage and retrieval.
2. Types of Document Loaders
  PyPDFLoader: Load PDF files page by page with metadata
  TextLoader: Load plain text files (.txt)
  CSVLoader: Load CSV files with row-based documents
  JSONLoader: Load JSON files with jq-style extraction
  UnstructuredLoader: Handle various file formats automatically
  DirectoryLoader: Load all files from a directory
  WebBaseLoader: Scrape and load web pages
  YouTubeLoader: Extract transcripts from YouTube videos {'cr

/var/folders/0t/c1wtjxgn4r72jg8lm3m6qz780000gn/T/ipykernel_7346/2424787457.py:7: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  results = retriever.get_relevant_documents(query)


##ETL

In [9]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

class PDFToChromaETL:
    def __init__(self, persist_dir="./chroma_db"):
        self.persist_dir = persist_dir
        self.embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

    def extract(self, pdf_path):                 # E
        return PyPDFLoader(pdf_path).load()

    def transform(self, docs):                   # T
        splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap = 150)
        return splitter.split_documents(docs)

         
    def load(self, chunks):                      # L
        return Chroma.from_documents(
            chunks,self.embeddings,
            persist_directory=self.persist_dir
        )

    def run(self, pdf_path):
        docs = self.extract(pdf_path)
        chunks = self.transform(docs)
        db = self.load(chunks)
        print(f"Loaded {len(chunks)} chunks")
        return db

# --- Run the full ETL job ---
etl = PDFToChromaETL()
db = etl.run("./data/langchain_demo.pdf")

# Verify: query the freshly built store
chunks = db.similarity_search(query, k=3)
for i, d in enumerate(chunks):
    print(f"\n chunk : {i} \n")
    print(d.metadata["page"], d.page_content)

Loaded 5 chunks

 chunk : 0 

0 LangChain Document Loaders - Demo Document
Understanding LangChain Document Loaders
1. Introduction
LangChain provides powerful document loaders that allow you to ingest data from various sources into your
LLM applications. Document loaders are essential for building RAG (Retrieval-Augmented Generation)
systems, chatbots, and knowledge bases.
This document serves as a demo file to test PDF loading capabilities in LangChain. When loaded, this
content will be split into chunks and can be used for vector storage and retrieval.
2. Types of Document Loaders
  PyPDFLoader: Load PDF files page by page with metadata
  TextLoader: Load plain text files (.txt)
  CSVLoader: Load CSV files with row-based documents
  JSONLoader: Load JSON files with jq-style extraction
  UnstructuredLoader: Handle various file formats automatically
  DirectoryLoader: Load all files from a directory
  WebBaseLoader: Scrape and load web pages
  YouTubeLoader: Extract transcripts from Y